# 04 · DeiT-Tiny for single-channel lensing maps

A Data-efficient Image Transformer (DeiT-Tiny, 5.7 M params, ImageNet-pretrained via `timm`)
adapted to the lensing data.

**Adaptations** (`deeplense/models/deit.py`)
1. **1-channel patch embedding**: the pretrained (192, 3, 16, 16) patch projection is collapsed
   to (192, 1, 16, 16) by summing over RGB (same argument as the EfficientNet stem).
2. **Native resolution instead of 224 upscaling**: 150 isn't divisible by the 16-px patch, so images are
   zero-padded (background intensity ≈ 0) to 160 → 10×10 = 100 tokens. timm resamples the
   pretrained 14×14 position embeddings to 10×10 bicubically. Compared to resizing to 224 this
   uses half the tokens (~2× cheaper) and never interpolates the substructure signal.
3. Stochastic depth 0.1, AdamW wd 0.05, 2 warm-up epochs, frozen-backbone first epoch,
   backbone LR = 0.4 × head LR. Transformers lack the CNN's locality prior, so the
   regularisation and pretrained initialisation matter more.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch

from deeplense import CLASS_NAMES
from deeplense.utils import get_device, seed_everything

seed_everything(42)
DEVICE = get_device()
DATA_ROOT = ROOT / "data" / "lensing"
RESULTS = ROOT / "results"
print("device:", DEVICE)

In [ ]:
# ── Run mode ──────────────────────────────────────────────────────────────────
# SMOKE = True : a few hundred images, 2 epochs, just to check everything runs (laptop).
# SMOKE = False: full dataset and full recipe (GPU recommended).
# If a full run already exists in results/<run>/ (e.g. from scripts/train.py),
# it is loaded instead of retraining unless RETRAIN = True.
SMOKE = True
RETRAIN = False

In [ ]:
from deeplense.models import DeiTTinyClassifier
m = DeiTTinyClassifier(pretrained=True)
print("patch embed:", m.backbone.patch_embed.proj)
print("tokens:", m.backbone.patch_embed.num_patches, "| pos_embed:", tuple(m.backbone.pos_embed.shape))

## Train (or load)

In [ ]:
from dataclasses import replace
from deeplense.data import DataConfig, build_loaders
from deeplense.metrics import predict
from deeplense.train import fit
from deeplense.utils import load_json
sys.path.insert(0, str(ROOT / "scripts"))
from train import RECIPES

cfg = RECIPES["deit"]
data_cfg = DataConfig(root=str(DATA_ROOT), num_workers=2)
if SMOKE:
    cfg = replace(cfg, epochs=2)
    data_cfg = replace(data_cfg, train_per_class=300, test_per_class=100)
RUN = "deit" + ("_smoke" if SMOKE else "")

loaders = build_loaders(data_cfg)
model = DeiTTinyClassifier(pretrained=True)
criterion = None
run_dir = RESULTS / RUN

if (run_dir / "best.pt").exists() and not RETRAIN:
    model.load_state_dict(torch.load(run_dir / "best.pt", map_location="cpu"))
    model.to(DEVICE)
    results = load_json(run_dir / "metrics.json")
    history = load_json(run_dir / "history.json")
    probs, labels = predict(model, loaders["test"], DEVICE)
    print(f"Loaded {run_dir}  (best epoch {results['best_epoch']})")
else:
    model, out = fit(model, loaders, cfg, DEVICE, criterion=criterion, out_dir=RESULTS, run_name=RUN)
    results, history, probs, labels = out, out["history"], out["probs"], out["labels"]

## Evaluation on the held-out test set

In [ ]:
from deeplense.metrics import classification_metrics, plot_confusion, plot_history, plot_roc

plot_history(history, title=RUN); plt.show()

m = classification_metrics(probs, labels)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_roc(probs, labels, title=RUN, ax=axes[0])
plot_confusion(m["confusion_matrix"], title="Test confusion matrix", ax=axes[1])
plt.tight_layout(); plt.savefig(RESULTS / RUN / "roc_confusion.png", dpi=130, bbox_inches="tight"); plt.show()

print(f"Test accuracy {m['accuracy']:.4f} | macro AUC {m['macro_auc']:.4f}")
for k, v in m["auc_per_class"].items():
    print(f"  {k:<20} AUC {v:.4f}")

## Attention rollout

Attention rollout (Abnar & Zuidema, 2020) propagates attention through all 12 layers to show
which patches the CLS token relies on. For a model that has learned the physics, attention
should concentrate on the Einstein ring and arcs, where substructure perturbations show up.

In [ ]:
import torch.nn.functional as F
test_ds = loaders["test"].dataset
fig, axes = plt.subplots(2, 3, figsize=(13, 8.5))
for c in range(3):
    i = next(j for j, y in enumerate(test_ds.labels) if y == c)
    x = test_ds[i][0][None].to(DEVICE)
    att = model.attention_rollout(x)[None]                       # (1, 1, 10, 10) over the padded 160x160
    att = F.interpolate(att, size=160, mode="bilinear")[0, 0, 5:155, 5:155].cpu()
    axes[0, c].imshow(x[0, 0].cpu(), cmap="inferno"); axes[0, c].set_title(CLASS_NAMES[c])
    axes[1, c].imshow(x[0, 0].cpu(), cmap="gray"); axes[1, c].imshow(att, cmap="jet", alpha=0.45)
    axes[1, c].set_title("attention rollout")
for ax in axes.flat: ax.axis("off")
plt.tight_layout(); plt.savefig(RESULTS / RUN / "attention_rollout.png", dpi=130, bbox_inches="tight"); plt.show()